# NeuroScan AI: Exploración de Datos, Manifiesto y Auditoría de Fuga
## Módulo 3: Databricks e IA Aplicada - Proyecto Final

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EduardCY/m3-proyecto-final-databricks-ia/blob/main/notebooks/01_exploracion_eda.ipynb)

Este notebook implementa la **Etapa 1** del pipeline:
1. Verificación de integridad y descarte de imágenes corruptas.
2. Generación del manifiesto estructurado con `pandas` y el paso equivalente en `PySpark`.
3. Auditoría empírica de **fuga de datos (data leakage)** entre particiones.
4. Creación del split estratificado (70% train, 15% val, 15% test) guardado en `data/manifest_split.csv`.


In [ ]:
# 0. Inicialización Automática de Entorno (Google Colab / Local)
import os
import sys

# Detección y preparación autónoma para Google Colab
if 'google.colab' in sys.modules:
    # 1. Clonar el repositorio público en /content si no existe
    if not os.path.exists('/content/m3-proyecto-final-databricks-ia'):
        print("🚀 Clonando repositorio NeuroScan AI en Google Colab...")
        !git clone https://github.com/EduardCY/m3-proyecto-final-databricks-ia.git /content/m3-proyecto-final-databricks-ia
    
    # 2. Posicionarse en la carpeta raíz del proyecto
    %cd /content/m3-proyecto-final-databricks-ia
    
    # 3. Instalar dependencias requeridas para Colab (MLflow, etc.)
    !pip install -q -r requirements-colab.txt

# 4. Asegurar que la raíz del proyecto está en el PYTHONPATH
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"✅ Entorno preparado en: {PROJECT_ROOT}")


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import CLASSES, DATA_DIR, MANIFEST_PATH, SPLIT_MANIFEST_PATH, SPLIT_RATIOS, SEED
from src.dataset import build_manifest, split_manifest, build_manifest_spark

print(f"Ruta de datos: {DATA_DIR}")
print(f"Clases a clasificar: {CLASSES}")


### 1. Construcción del Manifiesto con Auditoría de Integridad


In [ ]:
# Construir el manifiesto auditando imágenes corruptas
manifest_df = build_manifest(data_dir=DATA_DIR, use_cache=False)
print(f"Total imágenes válidas: {len(manifest_df)}")
manifest_df.head()


### 2. Análisis Exploratorio y Balance de Clases


In [ ]:
class_counts = manifest_df["label_name"].value_counts()
print("Distribución de clases:")
print(class_counts)

plt.figure(figsize=(8, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="viridis")
plt.title("Distribución de Clases de Tumores Cerebrales (MRI)")
plt.xlabel("Diagnóstico")
plt.ylabel("Cantidad de Cortes")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


### 3. Parser Distribuido Simbólico en PySpark (Rúbrica Databricks)


In [ ]:
# Demostración del procesamiento paralelo en PySpark
spark_df = build_manifest_spark(data_dir=DATA_DIR)
if spark_df is not None:
    print("Esquema inferido por Spark:")
    spark_df.printSchema()
    print("Conteo distribuido por clase:")
    spark_df.groupBy("label_name").count().show()
else:
    print("PySpark operando en modo fallback local (Java/Spark no inicializado en entorno ligero).")


### 4. Particionamiento Estratificado y Auditoría de Cero Fuga


In [ ]:
df_split = split_manifest(manifest_df, ratios=SPLIT_RATIOS, seed=SEED, output_path=SPLIT_MANIFEST_PATH)
print("Partición estratificada generada:")
print(df_split.groupby(["split", "label_name"]).size().unstack())

# Auditoría matemática de fuga de datos
train_paths = set(df_split[df_split["split"] == "train"]["path"])
val_paths = set(df_split[df_split["split"] == "val"]["path"])
test_paths = set(df_split[df_split["split"] == "test"]["path"])

assert len(train_paths.intersection(val_paths)) == 0, "¡FUGA DETECTADA entre Train y Val!"
assert len(train_paths.intersection(test_paths)) == 0, "¡FUGA DETECTADA entre Train y Test!"
assert len(val_paths.intersection(test_paths)) == 0, "¡FUGA DETECTADA entre Val y Test!"

print("\n✅ COMPROBACIÓN EXITOSA: Partición sin fuga de datos confirmada y guardada en data/manifest_split.csv")
